In [1]:
# ── CELL 1: Imports ───────────────────────────────
import pandas as pd
import numpy as np
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from aif360.algorithms.preprocessing import Reweighing
from aif360.algorithms.postprocessing import CalibratedEqOddsPostprocessing
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

print("All mitigation libraries imported successfully")

pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


All mitigation libraries imported successfully


In [2]:
# ── CELL 2: Reload Adult Dataset ──────────────────
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = ['age','workclass','fnlwgt','education','education-num',
           'marital-status','occupation','relationship','race',
           'sex','capital-gain','capital-loss','hours-per-week',
           'native-country','income']
df = pd.read_csv(url, names=columns, na_values=' ?',
                  skipinitialspace=True).dropna()

df['income_binary'] = (df['income']=='>50K').astype(int)
df['sex_binary']    = (df['sex']=='Male').astype(int)

cat_cols = ['workclass','education','marital-status','occupation',
            'relationship','race','native-country']
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))
df = df.drop(['income','sex','fnlwgt'], axis=1)

aif_adult = BinaryLabelDataset(
    df=df,
    label_names=['income_binary'],
    protected_attribute_names=['sex_binary'],
    favorable_label=1,
    unfavorable_label=0
)

privileged   = [{'sex_binary': 1}]
unprivileged = [{'sex_binary': 0}]

train, test = aif_adult.split([0.8], shuffle=True, seed=42)
print("Dataset ready for mitigation experiments")

Dataset ready for mitigation experiments


In [3]:
# ── CELL 2: Reload Adult Dataset ──────────────────
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = ['age','workclass','fnlwgt','education','education-num',
           'marital-status','occupation','relationship','race',
           'sex','capital-gain','capital-loss','hours-per-week',
           'native-country','income']
df = pd.read_csv(url, names=columns, na_values=' ?',
                  skipinitialspace=True).dropna()

df['income_binary'] = (df['income']=='>50K').astype(int)
df['sex_binary']    = (df['sex']=='Male').astype(int)

cat_cols = ['workclass','education','marital-status','occupation',
            'relationship','race','native-country']
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))
df = df.drop(['income','sex','fnlwgt'], axis=1)

aif_adult = BinaryLabelDataset(
    df=df,
    label_names=['income_binary'],
    protected_attribute_names=['sex_binary'],
    favorable_label=1,
    unfavorable_label=0
)

privileged   = [{'sex_binary': 1}]
unprivileged = [{'sex_binary': 0}]

train, test = aif_adult.split([0.8], shuffle=True, seed=42)
print("Dataset ready for mitigation experiments")

Dataset ready for mitigation experiments


In [4]:
# ── CELL 3: Baseline — Before Mitigation ──────────
model_base = XGBClassifier(random_state=42,
                            eval_metric='logloss', verbosity=0)
model_base.fit(train.features, train.labels.ravel())
y_pred_base = model_base.predict(test.features)

test_pred_base = test.copy()
test_pred_base.labels = y_pred_base.reshape(-1,1)

cm_base = ClassificationMetric(test, test_pred_base,
    privileged_groups=privileged,
    unprivileged_groups=unprivileged)

acc_base = cm_base.accuracy()
eod_base = cm_base.equal_opportunity_difference()
aod_base = cm_base.average_odds_difference()
spd_base = BinaryLabelDatasetMetric(aif_adult,
    privileged_groups=privileged,
    unprivileged_groups=unprivileged).statistical_parity_difference()

print("=== BEFORE MITIGATION (Baseline XGBoost) ===")
print(f"Accuracy:  {acc_base*100:.2f}%")
print(f"SPD:       {spd_base:.4f}")
print(f"EOD:       {eod_base:.4f}")
print(f"AOD:       {aod_base:.4f}")

=== BEFORE MITIGATION (Baseline XGBoost) ===
Accuracy:  87.10%
SPD:       -0.1963
EOD:       -0.0957
AOD:       -0.0835


In [5]:
# ── CELL 4: Pre-Processing — Reweighing ───────────
rw = Reweighing(
    unprivileged_groups=unprivileged,
    privileged_groups=privileged
)
train_rw = rw.fit_transform(train)

model_rw = XGBClassifier(random_state=42,
                          eval_metric='logloss', verbosity=0)
model_rw.fit(train_rw.features, train_rw.labels.ravel(),
             sample_weight=train_rw.instance_weights)

y_pred_rw = model_rw.predict(test.features)
test_pred_rw = test.copy()
test_pred_rw.labels = y_pred_rw.reshape(-1,1)

cm_rw = ClassificationMetric(test, test_pred_rw,
    privileged_groups=privileged,
    unprivileged_groups=unprivileged)

acc_rw  = cm_rw.accuracy()
eod_rw  = cm_rw.equal_opportunity_difference()
aod_rw  = cm_rw.average_odds_difference()

print("=== AFTER REWEIGHING (Pre-Processing) ===")
print(f"Accuracy:  {acc_rw*100:.2f}%")
print(f"EOD:       {eod_rw:.4f}")
print(f"AOD:       {aod_rw:.4f}")
print()
print(f"Accuracy change:  {(acc_rw-acc_base)*100:+.2f}%")
print(f"EOD improvement:  {abs(eod_base)-abs(eod_rw):+.4f}")
print(f"AOD improvement:  {abs(aod_base)-abs(aod_rw):+.4f}")

=== AFTER REWEIGHING (Pre-Processing) ===
Accuracy:  86.92%
EOD:       0.0812
AOD:       0.0294

Accuracy change:  -0.18%
EOD improvement:  +0.0145
AOD improvement:  +0.0540


In [6]:
# ── CELL 5: Post-Processing — Calibrated EO ───────
cpp = CalibratedEqOddsPostprocessing(
    privileged_groups=privileged,
    unprivileged_groups=unprivileged,
    cost_constraint='fnr',
    seed=42
)

# Need predict_proba — use Logistic Regression
from sklearn.linear_model import LogisticRegression
model_lr = LogisticRegression(random_state=42, max_iter=1000)
model_lr.fit(train.features, train.labels.ravel())

y_prob = model_lr.predict_proba(test.features)[:,1]
test_pred_lr = test.copy()
test_pred_lr.scores = y_prob.reshape(-1,1)
test_pred_lr.labels = model_lr.predict(test.features).reshape(-1,1)

cpp.fit(test_pred_lr, test)
test_pred_cpp = cpp.predict(test_pred_lr)

cm_cpp = ClassificationMetric(test, test_pred_cpp,
    privileged_groups=privileged,
    unprivileged_groups=unprivileged)

acc_cpp = cm_cpp.accuracy()
eod_cpp = cm_cpp.equal_opportunity_difference()
aod_cpp = cm_cpp.average_odds_difference()

cm_lr_base = ClassificationMetric(test, test_pred_lr,
    privileged_groups=privileged,
    unprivileged_groups=unprivileged)
acc_lr  = cm_lr_base.accuracy()
eod_lr  = cm_lr_base.equal_opportunity_difference()
aod_lr  = cm_lr_base.average_odds_difference()

print("=== AFTER CALIBRATED EQUALISED ODDS (Post-Processing) ===")
print(f"Accuracy:  {acc_cpp*100:.2f}%")
print(f"EOD:       {eod_cpp:.4f}")
print(f"AOD:       {aod_cpp:.4f}")
print()
print(f"Accuracy change:  {(acc_cpp-acc_lr)*100:+.2f}%")
print(f"EOD improvement:  {abs(eod_lr)-abs(eod_cpp):+.4f}")
print(f"AOD improvement:  {abs(aod_lr)-abs(aod_cpp):+.4f}")

=== AFTER CALIBRATED EQUALISED ODDS (Post-Processing) ===
Accuracy:  82.48%
EOD:       -0.3282
AOD:       -0.2062

Accuracy change:  -0.05%
EOD improvement:  -0.0161
AOD improvement:  -0.0083


In [7]:
# ── CELL 6: Before vs After Comparison Table ──────
comparison = pd.DataFrame([
    {'Approach': 'Baseline (XGBoost — no mitigation)',
     'Accuracy': f"{acc_base*100:.2f}%",
     'EOD': round(eod_base,4), 'AOD': round(aod_base,4)},
    {'Approach': 'Reweighing — Pre-Processing',
     'Accuracy': f"{acc_rw*100:.2f}%",
     'EOD': round(eod_rw,4),  'AOD': round(aod_rw,4)},
    {'Approach': 'Calibrated EO — Post-Processing',
     'Accuracy': f"{acc_cpp*100:.2f}%",
     'EOD': round(eod_cpp,4), 'AOD': round(aod_cpp,4)},
])

print("=== BIAS MITIGATION COMPARISON — Adult Dataset ===\n")
print(comparison.to_string(index=False))

comparison.to_csv(
    r'C:\Users\niyat\Desktop\pr\ML-Bias-Fairness\results\mitigation_comparison.csv',
    index=False)

print("\n✅ BIAS MITIGATION EXPERIMENTS COMPLETE")
print("\nThis is your accuracy vs fairness trade-off analysis")
print("These results go directly into Chapter 4 and Chapter 5")

=== BIAS MITIGATION COMPARISON — Adult Dataset ===

                          Approach Accuracy     EOD     AOD
Baseline (XGBoost — no mitigation)   87.10% -0.0957 -0.0835
       Reweighing — Pre-Processing   86.92%  0.0812  0.0294
   Calibrated EO — Post-Processing   82.48% -0.3282 -0.2062

✅ BIAS MITIGATION EXPERIMENTS COMPLETE

This is your accuracy vs fairness trade-off analysis
These results go directly into Chapter 4 and Chapter 5
